### **01 — Data Preparation**
This notebook:
- introduces the dataset and case-selection process
- checks files available for a selected case
- documents segmentation and mesh-export steps
- prepares inputs for mesh cleaning and skeletonisation

In [1]:
import os
import shutil
import nibabel as nib
import numpy as np

base_path = "/Users/ahthini/Documents//KCL/individual project/imageCAS_30_samples"
subject_folders = sorted(os.listdir(base_path))

for subject_folder in subject_folders:
    folder_path = os.path.join(base_path, subject_folder)
    if not os.path.isdir(folder_path):
        continue  #skip non-folder files
    
    print(f"\nProcessing: {subject_folder}")
    
    #label processing
    label_path = os.path.join(folder_path, "label.nii.gz")
    if os.path.exists(label_path):
        label_img = nib.load(label_path)
        label_data = label_img.get_fdata()

        label_data_16bit = label_data.astype(np.uint16)
        affine = label_img.affine.copy()

        #adjust geometry and set origin
        affine[:3, :3] /= 1000
        affine[:3, 3] = 0.0

        label_nii = nib.Nifti1Image(label_data_16bit, affine)
        label_nii.header.set_data_dtype(np.uint16)
        label_nii.set_qform(affine, code=1)
        label_nii.set_sform(affine, code=1)
        label_nii.header['qoffset_x'] = affine[0, 3]
        label_nii.header['qoffset_y'] = affine[1, 3]
        label_nii.header['qoffset_z'] = affine[2, 3]
        label_nii.header['sform_code'] = 1
        label_nii.header['qform_code'] = 1
        label_nii.header['intent_code'] = 0

        out_label_path = os.path.join(folder_path, "label_16bit.nii")
        nib.save(label_nii, out_label_path)
        print("✔ label_16bit.nii saved with origin zeroed")

    else:
        print(f"✘ label.nii.gz not found in {folder_path}")

    #image processing
    img_path = os.path.join(folder_path, "img.nii.gz")
    img_nii_file = os.path.join(folder_path, "img.nii")

    if os.path.exists(img_path):
        img = nib.load(img_path)
        nib.save(img, img_nii_file)
        print("✔ Copied and uncompressed img.nii.gz to img.nii")

    else:
        print(f"✘ img.nii.gz not found in {folder_path}")


Processing: 10064282
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10175956
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10257303
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10423186
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10746739
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10814698
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10878112
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 10974200
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.nii.gz to img.nii

Processing: 11089463
✔ label_16bit.nii saved with origin zeroed
✔ Copied and uncompressed img.n

### **Before continuing to notebook 02**
Notebook 01 prepares the CT images and coronary labels. Before mesh
cleaning and skeletonisation, the cardiac structures must be segmented
and the required surface meshes exported.

#### 1. Segment cardiac structures
Use TotalSegmentator with the `heartchambers_highres` task on each
patient's CT image (`img.nii`).

Retain the left ventricular myocardium and right ventricular
segmentations. The coronary artery segmentation is supplied separately
with the dataset; notebook 01 prepares it as `label_16bit.nii`.

#### 2. Inspect segmentations in ITK-SNAP
Open the CT image and overlay the corresponding segmentation.
Check that:
- the myocardial segmentation follows the left ventricular wall;
- the right ventricular segmentation matches the chamber;
- the supplied coronary label follows the visible coronary vessels;
- there are no obvious missing regions, disconnected branches, or
  major alignment errors.
Record any issues for the relevant patient before continuing.

#### 3. Export anatomical meshes
Using ITK-SNAP, export the surfaces and organise them under the
corresponding patient folder:
- Left ventricular myocardium: `myocardium.stl`
- Right ventricle: `heart/right_ventricle.stl`
- Coronary artery surface: `heart/label16mesh.vtk`

The coronary surface must be saved in VTK format for the existing
visualisation scripts. Renaming an STL file to `.vtk` does not convert
its format. Preserve the spatial coordinates during export so the anatomical
surfaces remain aligned.

#### 4. Check the patient folder
The relevant files should be arranged as follows:

```text
imageCAS_30_samples/
└── patient_id/
    ├── img.nii.gz
    ├── label.nii.gz
    ├── img.nii
    ├── label_16bit.nii
    ├── myocardium.stl
    └── heart/
        ├── right_ventricle.stl
        └── label16mesh.vtk
```

#### 5. Continue to notebook 02
Open `02_mesh_cleaning_and_skeletonisation.ipynb`.
The next notebook uses:
- `myocardium.stl` for myocardial mesh cleaning and quality checks;
- `label_16bit.nii` for coronary skeletonisation.

It produces:
- `myocardium_cleaned.stl`
- `myocardium_cleaned.vtp`
- `label_skeleton.nii`

The right ventricular and coronary surface meshes are used in later
alignment and visualisation steps.